# 04 — E-Perf-7: metering overhead decomposition

RFC-008 §E-Perf-7. Compares latency distributions across 4 metering
configurations: `fuel-only`, `epoch-only`, `neither` (baseline), and
`passthrough` (all metering enabled).

**Inputs**: `eval/results/e-perf-7/<host-tag>-<ts>/{fuel-only,epoch-only,neither,passthrough}/config-percentiles.json`
produced by `eval/scripts/summarise-e-perf-7.sh`.

**Configure** with `SHAKEDOWN_DIR` env var; otherwise newest shakedown is used.

In [ ]:
import json, os, glob
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

REPO = Path.cwd()
while REPO != REPO.parent and not (REPO / 'eval').is_dir():
    REPO = REPO.parent

shakedown_dir = os.environ.get('SHAKEDOWN_DIR')
if not shakedown_dir:
    batch_id = os.environ.get('WAFER_EVAL_BATCH_ID')
    if not batch_id:
        raise RuntimeError('SHAKEDOWN_DIR or WAFER_EVAL_BATCH_ID is required')
    shakedown_dir = REPO / 'eval/results/e-perf-7' / f'rpi5-{batch_id}'
shakedown_dir = Path(shakedown_dir)
print(f'shakedown: {shakedown_dir.relative_to(REPO)}')

In [ ]:
CONFIG_LABELS = ['neither', 'fuel-only', 'epoch-only', 'passthrough']
CONFIG_DISPLAY = {
    'neither': 'No metering',
    'fuel-only': 'Fuel only',
    'epoch-only': 'Epoch only',
    'passthrough': 'Both (default)',
}

rows = []
for label in CONFIG_LABELS:
    p = shakedown_dir / label / 'config-percentiles.json'
    if not p.exists():
        print(f'MISSING {p}')
        continue
    d = json.loads(p.read_text())
    agg = d['aggregate_over_runs']
    for pct in ['p50_ns', 'p95_ns', 'p99_ns', 'p999_ns']:
        a = agg.get(pct)
        if not a:
            continue
        rows.append({
            'config': label,
            'config_display': CONFIG_DISPLAY[label],
            'percentile': pct.rstrip('_ns'),
            'median_us': a['median'] / 1_000,
            'mean_us': a['mean'] / 1_000,
            'stdev_us': a['stdev'] / 1_000,
            'n': a['n'],
        })

df = pd.DataFrame(rows)
pivot = df.pivot(index='percentile', columns='config', values='median_us')[CONFIG_LABELS]
print('Median across runs (µs):')
print(pivot.round(1).to_string())

In [ ]:
# Compute deltas vs 'neither' baseline
baseline = df[df['config'] == 'neither'].set_index('percentile')['median_us']
delta_rows = []
for _, row in df.iterrows():
    b = baseline.get(row['percentile'], 0)
    delta_rows.append({
        'config': row['config'],
        'config_display': row['config_display'],
        'percentile': row['percentile'],
        'delta_us': row['median_us'] - b,
        'delta_pct': ((row['median_us'] - b) / b * 100) if b > 0 else 0,
    })
df_delta = pd.DataFrame(delta_rows)
pivot_delta = df_delta.pivot(index='percentile', columns='config', values='delta_us')[CONFIG_LABELS]
print('Delta vs no-metering baseline (µs):')
print(pivot_delta.round(2).to_string())

In [ ]:
# Bar chart: p50/p95/p99 comparison across configs
fig, ax = plt.subplots(figsize=(10, 6))

percentiles = ['p50', 'p95', 'p99']
x = np.arange(len(percentiles))
width = 0.2
colors = ['#2ecc71', '#3498db', '#e67e22', '#e74c3c']

for i, config in enumerate(CONFIG_LABELS):
    vals = []
    for pct in percentiles:
        row = df[(df['config'] == config) & (df['percentile'] == pct)]
        vals.append(row['median_us'].values[0] if len(row) > 0 else 0)
    ax.bar(x + i * width, vals, width, label=CONFIG_DISPLAY[config], color=colors[i])

ax.set_xlabel('Percentile')
ax.set_ylabel('Latency (µs)')
ax.set_title('E-Perf-7: metering overhead decomposition (Raspberry Pi 5 canonical)')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(percentiles)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Interpretation

On macOS M-series, metering overhead is negligible at the p50 level.
The epoch ticker (20 ms interval) imposes a small overhead because the
wasmtime epoch-check instruction fires on every loop back-edge. Fuel
accounting (per-instruction counter) shows near-zero cost because the
pass-through plugin executes very few instructions per call.

**Pi expectation**: The ARM Cortex-A76 has much slower branch prediction
and icache; epoch checks may show up more clearly. Fuel accounting should
remain negligible because the pass-through plugin's instruction count is
independent of architecture.